# Generating Benchmarking Circuits

This tutorial explains how to generate benchmarking circuits provided by quration. Quration generates the following circuits, provided a configuration file, and saves it in Quration-IR format, which is the intermediate language of Quration described later. The currently supported benchmark circuit generation is as follows:

- Quantum Phase Estimation based on Qubitization
  - PREPARE subroutine circuit
  - SELECT subroutine circuit
- Period Finding
  - Modular Bimultiply subroutine circuit
- Quantum dynamics simulation using Trotter decomposition
- Other Arithmetic circuits
  - QROM / UncomputeQROM
- Craig Adder / Cuccaro Adder

The following example calls the benchmark circuit generator obtained by compiling `quration-algorithm`. Therefore, please follow the compilation instructions in the `README.md` file. Please, make sure that an executable file such as `create_qpe` is generated in the `./build/benchmark_generators/` folder. We will temporarly add the `./build/benchmark_generators/` to the environment path, to allow the generators to be executed directly.

Moreover, the following tutorial commands uses the sample JSON files located at `quuration-algorithm/benchmark_generators/data`, which we will access with the `sample_data_dir` variable.

In [ ]:
import os
import pathlib
import platform

project_root = pathlib.Path("../../../..").resolve()
algorithm_generator_path = project_root / "build" / "benchmark_generators"
sample_data_dir = project_root / "quration-algorithm" / "benchmark_generators" / "data"
os.environ["PATH"] = str(algorithm_generator_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## Quantum Phase Estimation (QPE) / PREPARE / SELECT

Quantum Phase Estimation is an algorithm that samples the eigenvalues of a Hamiltonian represented by $H = \sum_i \alpha_i P_i$ using a probability distribution determined from a given initial state. The implementation is based on [R. Babbush et al., "Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity"](https://arxiv.org/abs/1805.03662) and implements the above by repeatedly applying subroutines called SELECT and PREPARE to the Hamiltonian block encoded using Linear Combination of Unitaries. To generate a QPE circuit, we need to run the `build/benchmark_generators/create_qpe` executable.

For example, if we want to estimate the phase for $H = 0.25 Z \otimes I + 0.25 Z \otimes I + 0.5 X \otimes X$, the json configuration file would look like:

```json
{
    "paulis": [
        ["Z", "I"],
        ["Z", "Z"],
        ["X", "X"]
    ],                                       // List of Pauli terms $P_i$ of the Hamiltonian
    "lcu_coefficients": [0.25, 0.25, 0.5],   // Pauli coefficients $\alpha_i$ of the Hamiltonian
    "system_size": 2,                        // Number of qubits of the Hamiltonian
    "hadamard_size": 3,                      // Fixed-point binary precision of the desired eigenvalue
    "sub_bit_precision": 3                   // Binary fixed-point precision when handling the coefficients alpha_i
}
```

In the following command we create a circuit with the input at `quration_algorithm/benchmark_generators/data/sample_qpe.json`. The command outputs a JSON file representing the intermediate representation (IR) of the QPE's circuit.

In [ ]:
!create_qpe --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_qpe.json"}

If we whish to only generate the PREPARE circuit that is used in QPE (the circuit corresponding to Fig. 11 in the above paper), run the `build/benchmark_generators/create_prepare` executable. The PREPARE circuit generates a state $U|0\rangle = \sum_i \alpha_i |i\rangle$, with positive coefficient $\alpha_i$ corresponding to the Hamiltonian $H = \sum_i \alpha_i P_i$. To generate PREPARE, the "lcu_coefficients" and "sub_bit_precision" input parameters are required from the QPE configuration.
If you run the command with the following options, a JSON file of the PREPARE circuit in the intermediate representation(IR) will be generated.

In [ ]:
!create_prepare --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_prepare.json"}

To generate the SELECT circuit used in QPE (corresponding to Fig. 5 and Fig. 7 in the above paper), we run the `build/benchmark_generators/create_select` executable. The SELECT circuit implements the unitary operation $U = \sum_i | i \rangle \langle i | \otimes P_i $ for a sequence of Pauli terms $P_i \in \pm \{I,X,Y,Z\}^{\otimes n}$, corresponding to the Hamiltonian is $H = \sum_i \alpha_i P_i$, with $ \alpha_i \ge 0$. Generating the SELECT circuit requires "pauli_strings" as the sole input parameter.


In [ ]:
!create_select --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_select.json"}

## Order-Finding / Modular Bimultiply

The Order-finding algorithm is a quantum algorithm that solves for an integer $r$ such that $x^r \equiv 1 (mod N)$, where $N$ an integer and $x$ is coprime to $N$.

In Shor's factoring algorithm, which finds the prime factors of the integer $N$. A value $x$ smaller than $N$ is chosen. If $x$ is not coprime with $N$, we can find a factor classically; otherwise, if $x$ is coprime to $N$, factorization is performed by using order-finding to discover the order ($r$) of $x$ modulo $N$.

In Quration, the circuit corresponding to Fig. 4 of [C.Gidney, "Factoring with n+2 clean qubits and n-1 dirty qubits"](https://arxiv.org/abs/1706.07884) is generated using `build/benchmark_generators/create_period_finding`.

For example, to solve the order-finding problem with at most a 4-bit value $r$ that satisfies $2^r \equiv 1 (mod 15)$, the input would be as follows.

```json
{
  "modulus": "15",         // modulus N
  "coprime_integer": "2",  // x (coprime to N)
  "depth": 4               // bit precision of the solution r
}
```

The following command outputs a JSON file of the order finder circuit in the intermediate representation. Here is an example command with the sample `quration_algorithm/benchmark_generators/data/sample_period_finding.json`.

In [ ]:
!create_period_finding --input {sample_data_dir / "sample_period_finding.json"} --output {output_dir / "tutorial_1_IR_period_finding.json"}

To generate the MultiControlledModBiMulImm circuit (see https://arxiv.org/pdf/1706.07884 Sec.2.2 and https://arxiv.org/abs/1905.07682), please run `build/benchmark_generators/create_multi_controlled_mod_bi_mul_imm`. This circuit is generated based on an integer $N$ and a multiplier $K$, and the circuit calculates $(xK, yK^{-1})\ mod N$ from two input integers $(x,y)$, using several control qubits. Here, the value $K$ is provided as an immediate hardcoded constant. As shown in Fig. 1 of https://arxiv.org/pdf/1706.07884, this circuit is the most important subroutine associated with Modular Exponentiation, used in order-finding.

The input contains the following parameters.  

```json
{
  "modulus": "15",          // modulus N
  "multiplier": "2",        // multiplier K
  "num_control_qubits": 1,  // number of control qubits
  "num_system_qubits": 4    // number of qubits to represent binary digits during calculations
}
```
Below there is an example command using the input at `quration_algorithm/benchmark_generators/data/sample_period_finding.json`.

Running the command with the following optional input parameters will output a JSON file of the MultiControlledModBiMulImm circuit in the intermediate representation.

In [ ]:
!create_multi_controlled_mod_bi_mul_imm --input {sample_data_dir / "sample_modular_bimultiply.json"} --output {output_dir / "tutorial_1_modular_bimultiply.json"}

## Quantum Dynamics Simulation (Trotter Expansion)

Quantum dynamics simulation is a quantum algorithm that simulates the quantum state $e^{iHt} |\psi_0 \rangle$, given the Hamiltonian $H=\sum_i \alpha_i P_i$, an initial state $|\psi_0 \rangle$, and a simulation time $t$. Quration implements an algorithm to approximate the simulation using Trotter expansion. In the Trotter expansion, for a sufficiently large integer $M$ called the Trotter number, we approximate the dynamics as $e^{iHt} = (e^{iHt/M})^M \simeq (\prod_i e^{i t \alpha_i P_i / M})^M$, and perform the above simulation by repeating infinitesimal Pauli infinitesimal rotation. To generate a circuit that simulates the time evolution of Hamiltonian using Trotter expansion, please run `build/benchmark_generators/create_trotter`.

We assume that the Hamiltonian is composed of Pauli strings. The input JSON file must contain the number of qubits, the Pauli strings, and their coefficients that make up the Hamiltonian. For example, the three-qubit longitudinal magnetic field Ising model can be written as follows.

```json
{
    "num_qubits": 3,                // Number qubits for the Hamiltonian
    "num_trotter_steps": 1,         // Trotter number (Trotter steps)
    "time": 1.0,                    // Simulated time
    "pauli_terms": [
        {
            "coeff": 1.0,
            "pauli_string": {
                "0": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "0": "Z",
                "1": "Z"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "1": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "1": "Z",
                "2": "Z"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "2": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "2": "Z",
                "0": "Z"
            }
        }
    ]                              // List of Pauli terms
}
```

Running the command will output a JSON file containing the circuit that simulates the time evolution of the specified Hamiltonian.
Below is an example command specifying the file `quration_algorithm/examples/data/1d_ising_hamiltonian.json`:

In [ ]:
!create_trotter --input {sample_data_dir / "sample_trotter.json"} --output {output_dir / "tutorial_1_trotter.json"}

## Quantum Read-Only Memory (QROM)

The Quantum Read-Only Memory (QROM) is an operation that loads pre-specified data corresponding to a given address. Specifically, for an integer $D_x$ associated with an address $x$, QROM is a routine that performs the transformation $U|x\rangle |0\rangle = |x\rangle |D_x\rangle$, where $|x\rangle$ is the address and the integer $D_x$ the data. This circuit is used as a subroutine in both the PREPARE circuit and Modular Bi-Multiply circuit. Uncompute QROM is a process that performs the inverse of this operation.

As an implementation example, Quration realizes QROM by loading data defined as $D_x = ax+b \mod 2^N$. Note that the circuit does not calculate the data at runtime; instead it simply reads the precomputed value $D_x$. Consequently, the values of $D_x$ will have no impact on the gate complexity or efficiency of the circuit. To generate a QROM circuit, please run `build/benchmark_generators/create_qrom`.

For example, if you want to build a QROM circuit with a 5-bit input address $x$ and the data $D_x$ defined as $D_x = 11x+3 \mod 2^6$, you can specify the following parameters:
```json
{
  "address_size": 5,           // Number of bits for the address x
  "value_size": 6,             // Number of bits for the data Dx
  "multiplier": "11",          // multiplier a
  "offset": "3"                // offset b
}
```
The creation command will output a JSON file with the specified QROM circuit.
Below is an example command with input `quration_algorithm/benchmark_generators/data/sample_qrom.json`.

In [ ]:
!create_qrom --input {sample_data_dir / "sample_qrom.json"} --output {output_dir / "tutorial_1_qrom.json"}

Similarly, if you want to generate the QROM uncompute circuit.

In [ ]:
!create_uncompute_qrom --input {sample_data_dir / "sample_qrom.json"} --output {output_dir / "tutorial_1_uncompute_qrom.json"}

## Craig Adder / Cuccaro Adder

Craig Adder and Cuccaro Adder are adder circuits that output $(x,x+y)$ for two input values $(x,y)$. See [C.Gidney, Halving the cost of quantum addition](https://arxiv.org/abs/1709.06648) for the details. The Craig Adder is known to be faster to execute in FTQC compared to the Cuccaro adder.

If you want to generate a 5-bit adder circuit, the JSON would look as followed.

```json
{
  "size": 5
}
```

The following command outputs a JSON file of Craig's Add circuit with the specified bit width. Here is an example command with the input JSON at `quration_algorithm/benchmark_generators/data/sample_add.json`.

In [ ]:
!create_add_craig --input {sample_data_dir / "sample_add.json"} --output {output_dir / "tutorial_1_add_craig.json"}

Similarly, the command below will output a JSON file of the cuccaro Adder circuit, given a specified bit width. In this example we use the input JSON located at `quration_algorithm/benchmark_generators/data/sample_add.json`.

In [ ]:
!create_add_cuccaro --input {sample_data_dir / "sample_add.json"} --output {output_dir / "tutorial_1_add_cuccaro.json"}